# Chapter 12: Stage 8 — Model Deployment, FastAPI Backend, Streamlit UI, Docker & MLOps

**Assalam-o-Alaikum!** Mubarak ho! Ab hum course ke aakhri aur sab se practical stage par hain: **Apne trained `.pkl` / `.pth` models ko FastAPI REST API, Streamlit Web App, aur Docker Container ke zariye poori duniya ke liye live deploy karna!**

### 1. Training Process vs Inference Deployment Mein Farq
- **Model Training (Offline / Scheduled):** Data load kar ke `model.fit()` chalana aur `.pkl` file banana. Yeh kaam offline ya hafta-waar schedule par hota hai.
- **Model Inference Server (Online 24/7 FastAPI):** Saved `.pkl` file ko memory mein 1 dafa load kar ke `POST /predict` endpoint ke zariye milliseconds mein predictions dena!

### 2. Full-Stack AI Application Architecture (`projects/student_prediction/`)
1. **Backend API (`fastapi_server.py`):** Pydantic schema validation ke saath `POST /predict` endpoint.
2. **Frontend UI (`streamlit_app.py`):** Sliders aur buttons wala web interface jo FastAPI ya pipeline ko call karta hai.
3. **Containerization (`Dockerfile`):** Application ko किसी bhi cloud server (Render, Railway, Hugging Face Spaces, AWS, GCP) par 1 command se chalane ke liye.

In [1]:
import os
import json

project_files = sorted(os.listdir("projects/student_prediction"))
print("Production Deployment Files in projects/student_prediction/:")
for f in project_files:
    print(" -", f)

Production Deployment Files in projects/student_prediction/:
 - Dockerfile
 - fastapi_server.py
 - predict_app.py
 - streamlit_app.py


### Step 1 — Inspect the Production FastAPI Server (`projects/student_prediction/fastapi_server.py`)
Yahan hum `Pydantic` (`BaseModel`, `Field`) se har input ki range (`study_hours_daily: 0..16`, `attendance_pct: 0..100`) validate karte hain taake koi ghalat request server ko crash na kar sake:

In [2]:
with open("projects/student_prediction/fastapi_server.py", "r", encoding="utf-8") as f:
    print(f.read())

#!/usr/bin/env python3
"""
FastAPI Production Inference Server for Student Final Marks Prediction
Run with: uvicorn projects.student_prediction.fastapi_server:app --reload --port 8000
"""

import os
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="Student Marks AI Prediction API", version="1.0.0")

MODEL_PATH = os.getenv("MODEL_PATH", "models/student_marks_pipeline.pkl")
pipeline = joblib.load(MODEL_PATH) if os.path.exists(MODEL_PATH) else None


class StudentInput(BaseModel):
    study_hours_daily: float = Field(..., ge=0.0, le=16.0, description="Daily study hours")
    attendance_pct: float = Field(..., ge=0.0, le=100.0, description="Attendance percentage")
    previous_score: float = Field(..., ge=0.0, le=100.0, description="Previous exam score")
    sleep_hours: float = Field(7.0, ge=3.0, le=12.0, description="Daily sleep hours")
    practice_tests: int = Field(3, ge=0, le=20, description="Pract

### Step 2 — Inspect the Streamlit Frontend (`projects/student_prediction/streamlit_app.py`) & `Dockerfile`

In [3]:
with open("projects/student_prediction/Dockerfile", "r", encoding="utf-8") as f:
    print("=== Dockerfile ===" + f.read())

=== Dockerfile ===FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY models/ ./models/
COPY projects/student_prediction/ ./projects/student_prediction/
EXPOSE 8000
CMD ["uvicorn", "projects.student_prediction.fastapi_server:app", "--host", "0.0.0.0", "--port", "8000"]



### How to Run & Deploy Locally or on Cloud
```bash
# 1. Start the FastAPI Inference Backend:
uvicorn projects.student_prediction.fastapi_server:app --reload --port 8000

# 2. Start the Streamlit Web UI:
streamlit run projects/student_prediction/streamlit_app.py

# 3. Or Build & Run with Docker:
docker build -t student-ai-app -f projects/student_prediction/Dockerfile .
docker run -p 8000:8000 student-ai-app
```

### Step 3 — Test the FastAPI Prediction Logic Directly in Python
Chalein dekhte hain ke jab कोई user ya frontend `POST /predict` par JSON bhejta hai toh hamara backend kaise validate aur predict karta hai:

In [4]:
import joblib
import pandas as pd

prod_pipe = joblib.load("models/student_marks_pipeline.pkl")

def simulate_fastapi_predict_endpoint(payload: dict):
    required = ["study_hours_daily", "attendance_pct", "previous_score", "sleep_hours", "practice_tests"]
    for col in required:
        if col not in payload:
            return {"status_code": 422, "error": f"Missing required field: {col}"}
    if not (0.0 <= payload["study_hours_daily"] <= 16.0):
        return {"status_code": 422, "error": "study_hours_daily must be between 0 and 16"}
    df_in = pd.DataFrame([payload])[required]
    score = round(max(0.0, min(100.0, float(prod_pipe.predict(df_in)[0]))), 2)
    return {
        "status_code": 200,
        "predicted_final_marks": score,
        "status": "PASS" if score >= 50.0 else "AT_RISK",
    }

print("Valid Request (200 OK):", simulate_fastapi_predict_endpoint({
    "study_hours_daily": 7.0, "attendance_pct": 94.0, "previous_score": 84.0, "sleep_hours": 7.5, "practice_tests": 4
}))
print("Invalid Request (422 Unprocessable):", simulate_fastapi_predict_endpoint({
    "study_hours_daily": 25.0, "attendance_pct": 94.0, "previous_score": 84.0, "sleep_hours": 7.5, "practice_tests": 4
}))

Valid Request (200 OK): {'status_code': 200, 'predicted_final_marks': 100.0, 'status': 'PASS'}
Invalid Request (422 Unprocessable): {'status_code': 422, 'error': 'study_hours_daily must be between 0 and 16'}


### Step 4 — Production Logging & Model Drift Monitoring
Deploy karne ke baad MLOps ka sab se ahem hissa **Prediction Logging & Monitoring** hai — har prediction ka timestamp, input summary, aur latency log karein taake agar model ki performance waqt ke saath gire toh foran pata chal jaye:

In [5]:
import time

def logged_inference(payload: dict):
    t0 = time.time()
    resp = simulate_fastapi_predict_endpoint(payload)
    latency_ms = round((time.time() - t0) * 1000, 3)
    log_entry = {
        "latency_ms": latency_ms,
        "input_study_hours": payload.get("study_hours_daily"),
        "output": resp,
    }
    return log_entry

print("Production Inference Log Entry:", logged_inference({
    "study_hours_daily": 5.5, "attendance_pct": 85.0, "previous_score": 72.0, "sleep_hours": 7.0, "practice_tests": 3
}))

Production Inference Log Entry: {'latency_ms': 0.996, 'input_study_hours': 5.5, 'output': {'status_code': 200, 'predicted_final_marks': 86.99, 'status': 'PASS'}}


### Final Course Summary & Next Steps for Your Students
**Mubarak ho!** Is 12-Chapter Course ke zariye aap ke students ne seekh liya:
1. **Stage 1 & 2 (Machine Learning):** Raw Excel/CSV/JSON data ko clean karna, leak-free `Pipeline` banana, Regression aur Classification metrics (`MAE`, `R2`, `Confusion Matrix`, `Precision`, `Recall`, `F1`, `ROC-AUC`), Overfitting rokna, aur `GridSearchCV` se hyperparameter tuning karna.
2. **Stage 3 & 4 (Deep Learning & Computer Vision):** Neural Networks (`Forward Pass`, `Loss`, `Backpropagation`), **PyTorch (`nn.Module`)**, aur Pretrained **Transfer Learning (`ResNet18`)** se image classification.
3. **Stage 5, 6 & 7 (NLP, Generative AI, RAG & AI Agents):** `TF-IDF` sentiment analysis, Word Embeddings, Self-Attention, Structured JSON outputs, **Vector Search RAG Assistants**, aur **Tool-Using AI Agents with Human Approval Guardrails**.
4. **Stage 8 (Deployment & MLOps):** **FastAPI** backend, **Streamlit** UI, **Docker** containerization, aur production monitoring!